# Number of Target Estimation with AIC and MDL



In [ ]:
# Importing modules
import numpy as np;
import matplotlib.pyplot as plt;

In [ ]:
# Defining functions
def g_mean(arr):
  g = 1; 
  for i in range(len(arr)):
    g = g*arr[i]; 
  return g**(1/len(arr)); 


def a_mean(arr):
  a = 0; 
  for i in range(len(arr)):
    a = a+arr[i]; 
  return a/len(arr); 


def steering_vector(
    sensor_pos: np.ndarray,
    ang_elev: float,
    ang_azim: float,
    wavelen: float
    ) -> np.ndarray:
  omega = np.array([[np.sin(ang_elev)*np.cos(ang_azim)],
                    [np.sin(ang_elev)*np.sin(ang_azim)],
                    [np.cos(ang_elev)]]);
  return np.exp(1j*2*np.pi/wavelen*sensor_pos.T@omega); 


def generate_pos_1d_ula(
    N: int,
    d: float,
    axis=(1.,0.,0.),
    x_init=(0.,0.,0.)
    ) -> np.ndarray:
  if len(axis) != 3:
    raise TypeError(f"""The axis argument represents a 3D Cartesian vector.
      The length of the input ({len(axis)}) does not match with expected
      size 3."""); 
  if sum(axis) != 1:
    axis_new = (x/sum(axis) for x in axis); 
    axis = axis_new; 

  if len(x_init) != 3:
    raise TypeError(f"""The x_init argument represents a 3D cartesian vector.
      The length of the input ({len(x_init)}) does not match with expected
      size 3."""); 

  sensor_pos = np.tile(x_init, N).reshape(N,3).T \
    + (np.arange(0,N,1)*d) * np.tile(np.array(axis), N).reshape(N,3).T; 
  return sensor_pos; 


def generate_random_targets(ang_min: float, ang_max: float, ang_dist: float, K: int) -> np.ndarray:
  while True:
    angs = np.random.uniform(ang_min, ang_max, K, dtype=np.float64); 
    angs_sort = np.sort(angs); 
    for i in range(K-1):  # break for if is not valid
      is_valid = np.abs(angs_sort[i+1] - angs_sort[i]) >= ang_dist; 
      if not is_valid:
        break; 
    if is_valid: break;   # break while if is valid
  return angs; 

In [ ]:
# Defining parameters
c = 3*1e8;      # The speed of light in vacuum (m/s)
f = 5*1e9;      # Narrowband signal frequency (Hz)
wl = c/f;       # Signal wavelenght (m)
d = wl/2;       # Antenna distance (m)

N = 16;         # Number of antennas
T = 100;        # Number of snapshots
K = np.random.randint(1,N/2);          # Number of targets (selected randomly between 1 and N/2)
symmetrization = False

S_db = [0]*K;   # Target signal power (dB)
snr_db = 0.0;   # Signal-to-Noise Ratio (dB)

ang_azim = 0;   # Azimuth angle (deg)
ang_min = -60;  # Minimum elevation angle (deg)
ang_max = 60;   # Maximum elevation angle (deg)
ang_dist = 5;   # Minimum distance between targets (deg)
ang_res = 0.1   # Angular resolution in scanning (deg)

sensor_pos = generate_pos_1d_ula(N, d);                                 # Sensor position
theta_scan = np.arange(-90,90+ang_res,ang_res);                         # Angle scan
true_angles = generate_random_targets(ang_min, ang_max, ang_dist, K);   # Target elevation angles

for k in range(K):
  print(f"Target {k} True Angle: {true_angles[k]:.3f}°"); 

In [ ]:
# Target Angle Generation
while True:
  true_angles = np.random.uniform(ang_min, ang_max, K);
  angles_sort = np.sort(true_angles)
  for i in range(K-1): # break for if is not valid
    is_valid = np.abs(angles_sort[i+1] - angles_sort[i]) >= ang_dist;
    if not is_valid:
      break;
  if is_valid: break; # break while if is valid

for k in range(K):
  print(f"Target {k} True Angle: {true_angles[k]:.2f}°");

Target 0 True Angle: -36.04°
Target 1 True Angle: 10.31°
Target 2 True Angle: -8.49°
Target 3 True Angle: 2.53°
Target 4 True Angle: 57.90°
Target 5 True Angle: 38.10°
Target 6 True Angle: 84.14°


In [ ]:
# Data Generation
A = np.column_stack([steering_vector(sensor_pos, np.deg2rad(theta), np.deg2rad(0), wl) for theta in true_angles]); # Array Steering Matrix
S = (np.random.randn(K, T) + 1j*np.random.randn(K, T))/np.sqrt(2); # Target Signals: Uncorrelated Gaussian (variance = 1.0)

# Noise: Spatially white complex Gaussian noise
noise_var = 10**(-snr_db/10); 
Noise = np.sqrt(noise_var) * (np.random.randn(N, T) + 1j*np.random.randn(N, T))/np.sqrt(2); 

# Received Signal at the Array
X = A @ S + Noise; 

In [ ]:
Rxx_samp = (X @ X.conj().T)/T;        # Sample (estimated) covariance
eig_val, eig_vec = np.linalg.eigh(Rxx_samp); 

In [ ]:
Rxx_samp = (X @ X.conj().T)/T;        # Sample (estimated) covariance
if symmetrization:                    # Hermitian Symmetrization guarantees Hermitian matrix for lower snapshot scenarios.
  Rxx_samp = 0.5*(Rxx_samp + Rxx_samp.conj().T); 
eig_val, eig_vec = np.linalg.eigh(Rxx_samp); 

## Akaike Information Criterion (AIC)

In [ ]:
# Akaike Information Criterion (AIC)
aic = np.zeros(N); 
for k in range(N):
  aic[k] = -2*T*(N-k+1)*np.log(g_mean(eig_val[:N-k])/a_mean(eig_val[:N-k])) + 2*k*(2*N-k); 

## Minimum Description Length (MDL)

In [ ]:
# Minimum Description Length (MDL)
mdl = np.zeros(N); 
for k in range(N):
  mdl[k] = -T*(N-k+1)*np.log(g_mean(eig_val[:N-k])/a_mean(eig_val[:N-k])) + k*(2*N-k)*np.log(T)/2; 

In [ ]:
K_aic = np.argmin(aic); 
K_mdl = np.argmin(mdl); 

print(f"True number of targets: {K}"); 
print(f"Akaike Information Criterion estimation: {K_aic}"); 
print(f"Minimum Description Length estimation: {K_mdl}"); 

True number of targets: 7
Akaike Information Criterion estimation: 7
Minimum Description Length estimation: 7
